# Protein Checkpoint Top-1 Benchmark

Evaluate a protein ProGen checkpoint on held-out `train.txt` lines with causal-LM loss and next-token top-1 accuracy.

In [ ]:
import json
import math
import sys
from pathlib import Path

import torch

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "libs").is_dir():
            return path
    raise RuntimeError("Could not locate project root from the current notebook directory.")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from libs.core import (
    MDCDecoderModel,
    MDCModelConfig,
    compute_mdc_causal_lm_loss,
    create_protein_lm_dataloader,
    discover_protein_train_text_paths,
    is_supported_protein_checkpoint_family,
    load_protein_corpus_text_parts,
    load_protein_pretrain_checkpoint,
    split_protein_corpus_text,
)
from libs.data.training.tokenizer import SequenceTokenizer

PROJECT_ROOT

In [ ]:
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "checkpoints" / "protein_from_scratch"
CHECKPOINT_PATH = CHECKPOINT_DIR / "checkpoint_best.pt"
if not CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH = CHECKPOINT_DIR / "checkpoint_last.pt"

TRAIN_RATIO = 0.9
BATCH_SIZE = 4
MAX_BATCHES = 100
RESULTS_PATH = PROJECT_ROOT / "data" / "evals" / "protein_top1_benchmark.json"

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Missing checkpoint: {CHECKPOINT_PATH}")

checkpoint_meta = torch.load(CHECKPOINT_PATH, map_location="cpu")
if not is_supported_protein_checkpoint_family(checkpoint_meta.get("model_family")):
    raise ValueError(
        "This benchmark expects a stage-2 protein pretrain checkpoint. "
        f"Received: {checkpoint_meta.get('model_family')!r}"
    )

checkpoint_meta.keys()

In [ ]:
tokenizer_map_path = Path(checkpoint_meta["tokenizer_map_path"])
if not tokenizer_map_path.exists():
    tokenizer_map_path.parent.mkdir(parents=True, exist_ok=True)
    tokenizer_map_path.write_text(
        json.dumps(checkpoint_meta["tokenizer_map"], ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

tokenizer = SequenceTokenizer.load_map(tokenizer_map_path)
train_text_path = Path(
    checkpoint_meta.get("corpus_file")
    or PROJECT_ROOT / "data" / "compiled" / "refseq_bacteria_protein" / "train.txt"
)
checkpoint_corpus_files = tuple(Path(path) for path in checkpoint_meta.get("corpus_files", ()) if Path(path).exists())
if checkpoint_corpus_files:
    train_text_paths = checkpoint_corpus_files
else:
    train_text_paths = discover_protein_train_text_paths(
        train_text_path,
        part_glob="train_part_*.txt",
        prefer_parts=True,
    )
corpus_text = load_protein_corpus_text_parts(train_text_paths)
_, val_text = split_protein_corpus_text(corpus_text, train_ratio=TRAIN_RATIO)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_config_payload = dict(checkpoint_meta["model_config"])
if model_config_payload.get("layer_types") is not None:
    model_config_payload["layer_types"] = tuple(model_config_payload["layer_types"])
if device.type != "cuda":
    model_config_payload["dtype"] = torch.float32

model_config = MDCModelConfig(**model_config_payload)
context_length = int(model_config.context_length)
val_loader = create_protein_lm_dataloader(
    val_text,
    tokenizer,
    context_length=context_length,
    stride=max(1, context_length // 2),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

model = MDCDecoderModel(model_config).to(device)
checkpoint = load_protein_pretrain_checkpoint(
    CHECKPOINT_PATH,
    model=model,
    optimizer=None,
    map_location=device,
)

print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Corpus: {train_text_path}")
print(f"Corpus files: {len(train_text_paths)}")
print(f"Tokenizer vocab: {tokenizer.vocab_size}")
print(f"Validation batches: {len(val_loader)} device={device}")

In [ ]:
model.eval()
losses = []
correct = 0
total = 0

with torch.no_grad():
    for batch_index, batch in enumerate(val_loader):
        if batch_index >= MAX_BATCHES:
            break

        batch = type(batch)(
            input_ids=batch.input_ids.to(device),
            attention_mask=batch.attention_mask.to(device),
            labels=batch.labels.to(device),
        )
        logits = model(batch.input_ids, attn_mask=batch.attention_mask)
        loss = compute_mdc_causal_lm_loss(logits, batch.labels)
        losses.append(float(loss.item()))

        predictions = logits.argmax(dim=-1)
        valid_mask = batch.labels != -100
        correct += int((predictions[valid_mask] == batch.labels[valid_mask]).sum().item())
        total += int(valid_mask.sum().item())

mean_loss = sum(losses) / len(losses) if losses else float("nan")
results = {
    "checkpoint_path": str(CHECKPOINT_PATH.resolve()),
    "corpus_file": str(train_text_path.resolve()),
    "model_family": checkpoint.get("model_family"),
    "backbone_family": checkpoint.get("backbone_family"),
    "batches_evaluated": len(losses),
    "tokens_evaluated": total,
    "loss": mean_loss,
    "perplexity": math.exp(mean_loss) if math.isfinite(mean_loss) else float("nan"),
    "top1_accuracy": correct / total if total else float("nan"),
}

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.write_text(json.dumps(results, indent=2) + "\n", encoding="utf-8")
results